# EXP-009: Does the core pattern hold under Adam, not just plain SGD? (R4)

**Status: engineering validation / proof of concept**, same caveats as all prior
notebooks this session -- not the frozen Stage D experiment.

**Purpose (R4 of the revision roadmap, Perspective/Domain/Methodology reviewers):**
every result so far (EXP-001 through EXP-008) used plain online SGD. Reviewers
flagged that plain SGD is an unusual choice for anything resembling a realistic
training setup, and that an adaptive optimizer (Adam) could behave very
differently under LR-scaling interventions -- Adam already adapts its effective
per-parameter step size from gradient second-moment estimates, so multiplying
the base LR by a fixed factor on "reduce"/"increase" may have a much smaller (or
larger, or differently-shaped) effect than it does under plain SGD. This
notebook re-runs the CORE comparisons from EXP-001 side-by-side under SGD and
Adam, using the SAME task/disturbance/controller code, to test whether the two
headline findings are optimizer-specific artifacts or genuinely general:
1. Does Arm D (anticipatory) still trigger earlier than Arm C (reactive) under
   forecastable disturbances?
2. Does "increase" still win for drift and "reduce" still win for gradient-noise
   instability (the core diagnosis-must-match-disturbance claim)?
3. Does `auto_grad` (gradient-norm-elevation-vs-loss-elevation) still correctly
   diagnose which is which?

**Calibration note (carries forward the R3 fix):** the intervention threshold and
baseline statistics are calibrated from a POST-CONVERGENCE window (steps
150-199, immediately before DISTURBANCE_ONSET=200), not the non-stationary
training-transient warmup window EXP-001 originally used -- see R3's changelog
entry in AGENT.md for why the original warmup-window calibration was ~9x too
loose and, when corrected, exposed a near-total false-positive-rate problem in
the trigger design itself (still unresolved, flagged separately). Using the
corrected calibration here from the start avoids re-confounding this optimizer
comparison with that already-known calibration issue.

**Design choice for Adam's base LR:** Adam's effective step size is
self-normalizing (roughly independent of gradient scale, unlike SGD), so the
SGD-tuned base_lr=0.1 is not a like-for-like comparison -- 0.1 is far too large
for Adam on this toy net (would diverge/oscillate). base_lr=0.01 was chosen as a
conventional, untuned "reasonable default" for Adam on a small MLP -- explicitly
NOT tuned to make the results look good, kept fixed across all conditions and
seeds, same spirit as SGD's constants throughout this session.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

torch.set_num_threads(1)
print("torch", torch.__version__, "| numpy", np.__version__)


C:\Users\USER\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch 2.5.1+cpu | numpy 2.0.2


## 1. Toy task, disturbances, controllers (same as EXP-001/EXP-002, R3-corrected calibration)

In [2]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

MEAN0_BASE = np.array([-1.0, -1.0])
MEAN1_BASE = np.array([1.0, 1.0])
BASE_NOISE_SCALE = 0.5

def gen_batch(rng, drift_frac=0.0, batch_size=16):
    frac = float(np.clip(drift_frac, 0.0, 1.0))
    mean0 = (1 - frac) * MEAN0_BASE + frac * MEAN1_BASE
    mean1 = (1 - frac) * MEAN1_BASE + frac * MEAN0_BASE
    y = rng.integers(0, 2, size=batch_size)
    means = np.where(y[:, None] == 0, mean0, mean1)
    x = means + rng.normal(scale=BASE_NOISE_SCALE, size=(batch_size, 2))
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += float((p.grad.detach() ** 2).sum())
    return total ** 0.5

N_STEPS = 400
DISTURBANCE_ONSET = 200
RAMP_LEN = 40
GRAD_NOISE_MAX = 8.0
STEADY_STATE_WINDOW = 100
TREND_WINDOW = 5
FORECAST_HORIZON = 15
COOLDOWN = 30
LR_REDUCTION_FACTOR = 0.3
LR_BOOST_FACTOR = 3.0

# R3-corrected calibration window (post-convergence, immediately pre-onset)
CALIB_START = 150
CALIB_END = 200

def _ramp(step):
    if step < DISTURBANCE_ONSET:
        return 0.0
    return min(step - DISTURBANCE_ONSET, RAMP_LEN) / RAMP_LEN

def _instant(step):
    return 1.0 if step >= DISTURBANCE_ONSET else 0.0

def drift_forecastable(step):
    return (_ramp(step), 0.0)

def drift_unforecastable(step):
    return (_instant(step), 0.0)

def instability_forecastable(step):
    return (0.0, GRAD_NOISE_MAX * _ramp(step))

def instability_unforecastable(step):
    return (0.0, GRAD_NOISE_MAX * _instant(step))

def trend_forecast(history, horizon=1):
    n = len(history)
    if n < 2:
        return history[-1] if n else 0.0
    x = np.arange(n)
    y = np.array(history)
    slope, intercept = np.polyfit(x, y, 1)
    target_x = n - 1 + horizon
    return float(slope * target_x + intercept)

GRAD_RATIO_SENSITIVITY = 1.5

def auto_select_intervention_grad(loss_window, gradnorm_window, baseline_loss_mean, baseline_gradnorm_mean,
                                   sensitivity=GRAD_RATIO_SENSITIVITY):
    loss_ratio = (float(np.mean(loss_window)) + 1e-6) / (baseline_loss_mean + 1e-6)
    gradnorm_ratio = (float(np.mean(gradnorm_window)) + 1e-6) / (baseline_gradnorm_mean + 1e-6)
    return "reduce" if gradnorm_ratio > sensitivity * loss_ratio else "increase"


## 2. Single training run, parameterized by optimizer

In [3]:
def make_optimizer(optimizer_name, params, base_lr):
    if optimizer_name == "sgd":
        return torch.optim.SGD(params, lr=base_lr)
    elif optimizer_name == "adam":
        return torch.optim.Adam(params, lr=base_lr)
    raise ValueError(optimizer_name)

BASE_LR = {"sgd": 0.1, "adam": 0.01}

def run_trial(seed, arm, disturbance_fn, intervention_type, optimizer_name):
    '''arm: 'C' (reactive) or 'D' (anticipatory). intervention_type: 'reduce', 'increase',
    or 'auto_grad'. optimizer_name: 'sgd' or 'adam'. Same task/controller logic as
    EXP-001, with the R3-corrected calibration window (CALIB_START:CALIB_END).'''
    assert arm in ("C", "D")
    assert intervention_type in ("reduce", "increase", "auto_grad")
    assert optimizer_name in ("sgd", "adam")
    rng = np.random.default_rng(seed)
    model = make_model(seed)
    base_lr = BASE_LR[optimizer_name]
    opt = make_optimizer(optimizer_name, model.parameters(), base_lr)
    loss_fn = nn.BCEWithLogitsLoss()

    loss_hist, gradnorm_hist = [], []
    threshold = baseline_loss_mean = baseline_gradnorm_mean = None
    trigger_step = None
    cooldown_remaining = 0
    active_intervention = None

    for step in range(N_STEPS):
        drift_frac, grad_noise_extra = disturbance_fn(step)
        xb, yb = gen_batch(rng, drift_frac=drift_frac)

        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()

        loss_t = float(loss.item())
        if grad_noise_extra > 0:
            with torch.no_grad():
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * grad_noise_extra)
        gn_t = grad_norm(model)

        loss_hist.append(loss_t)
        gradnorm_hist.append(gn_t)

        if step == CALIB_END - 1:
            calib_loss = np.array(loss_hist[CALIB_START:CALIB_END])
            threshold = float(calib_loss.mean() + 3 * calib_loss.std() + 1e-6)
            baseline_loss_mean = float(calib_loss.mean())
            baseline_gradnorm_mean = float(np.mean(gradnorm_hist[CALIB_START:CALIB_END]))

        if step >= CALIB_END and trigger_step is None and threshold is not None:
            if arm == "C":
                signal = loss_t
            else:
                window = loss_hist[max(0, step - TREND_WINDOW + 1):step + 1]
                signal = trend_forecast(window, horizon=FORECAST_HORIZON)
            if signal > threshold:
                trigger_step = step
                cooldown_remaining = COOLDOWN
                loss_window = loss_hist[max(0, step - TREND_WINDOW + 1):step + 1]
                if intervention_type == "auto_grad":
                    gradnorm_window = gradnorm_hist[max(0, step - TREND_WINDOW + 1):step + 1]
                    active_intervention = auto_select_intervention_grad(
                        loss_window, gradnorm_window, baseline_loss_mean, baseline_gradnorm_mean)
                else:
                    active_intervention = intervention_type

        if cooldown_remaining > 0:
            factor = LR_REDUCTION_FACTOR if active_intervention == "reduce" else LR_BOOST_FACTOR
            for g in opt.param_groups:
                g["lr"] = base_lr * factor
            cooldown_remaining -= 1
        else:
            for g in opt.param_groups:
                g["lr"] = base_lr

        opt.step()

    loss_arr = np.array(loss_hist)
    post_onset = loss_arr[DISTURBANCE_ONSET:]
    peak_loss = float(post_onset.max())
    steady_state_loss = float(np.mean(loss_arr[-STEADY_STATE_WINDOW:]))

    return {
        "seed": seed, "arm": arm, "intervention_type": intervention_type,
        "optimizer": optimizer_name, "active_intervention": active_intervention,
        "trigger_step": trigger_step, "threshold": threshold,
        "peak_loss": peak_loss, "steady_state_loss": steady_state_loss,
    }


## 3. Run all optimizers x conditions x arms x intervention types x seeds

In [4]:
SEEDS = list(range(10))
CONDITIONS = {
    "drift_forecastable": drift_forecastable, "drift_unforecastable": drift_unforecastable,
    "instability_forecastable": instability_forecastable, "instability_unforecastable": instability_unforecastable,
}
INTERVENTION_TYPES = ("reduce", "increase", "auto_grad")
OPTIMIZERS = ("sgd", "adam")

results = []
for optimizer_name in OPTIMIZERS:
    for cond_name, fn in CONDITIONS.items():
        for arm in ("C", "D"):
            for itype in INTERVENTION_TYPES:
                for seed in SEEDS:
                    r = run_trial(seed, arm, fn, itype, optimizer_name)
                    lead_time = (r["trigger_step"] - DISTURBANCE_ONSET) if r["trigger_step"] is not None else None
                    results.append({
                        "optimizer": optimizer_name, "condition": cond_name, "arm": arm,
                        "intervention": itype, "seed": seed, "trigger_step": r["trigger_step"],
                        "lead_time": lead_time, "peak_loss": r["peak_loss"],
                        "steady_state_loss": r["steady_state_loss"],
                        "active_intervention": r["active_intervention"],
                    })

df = pd.DataFrame(results)
print(f"Total trials: {len(df)}")
df.head(10)


Total trials: 480


,optimizer,condition,arm,intervention,seed,trigger_step,lead_time,peak_loss,steady_state_loss,active_intervention
0,sgd,drift_forecastable,C,reduce,0,206.0,6.0,3.280475,0.049026,reduce
1,sgd,drift_forecastable,C,reduce,1,207.0,7.0,3.246233,0.046755,reduce
2,sgd,drift_forecastable,C,reduce,2,203.0,3.0,3.115008,0.060225,reduce
3,sgd,drift_forecastable,C,reduce,3,200.0,0.0,2.689970,0.064077,reduce
4,sgd,drift_forecastable,C,reduce,4,206.0,6.0,3.110644,0.034735,reduce
5,sgd,drift_forecastable,C,reduce,5,204.0,4.0,3.302267,0.051998,reduce
6,sgd,drift_forecastable,C,reduce,6,205.0,5.0,3.447991,0.043601,reduce
7,sgd,drift_forecastable,C,reduce,7,203.0,3.0,3.032062,0.050537,reduce
8,sgd,drift_forecastable,C,reduce,8,207.0,7.0,3.078520,0.051582,reduce
9,sgd,drift_forecastable,C,reduce,9,210.0,10.0,3.262336,0.047002,reduce


## 4. Summary and the three headline checks, SGD vs Adam side by side

In [5]:
summary = df.groupby(["optimizer", "condition", "arm", "intervention"]).agg(
    n_triggered=("trigger_step", lambda s: s.notna().sum()),
    mean_lead_time=("lead_time", "mean"),
    mean_peak_loss=("peak_loss", "mean"), std_peak_loss=("peak_loss", "std"),
    mean_steady_state_loss=("steady_state_loss", "mean"), std_steady_state_loss=("steady_state_loss", "std"),
).round(4)

def get(opt_name, cond, arm, itype, col):
    return summary.loc[(opt_name, cond, arm, itype), col]

summary


n_triggered  \
optimizer condition                  arm intervention                
adam      drift_forecastable         C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
          drift_unforecastable       C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
          instability_forecastable   C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
          instability_unforecastable C   auto_grad               9   
                                         increase                9   
                                         reduce                  9   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
sgd       drift_forecastable         C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
          drift_unforecastable       C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
          instability_forecastable   C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
          instability_unforecastable C   auto_grad              10   
                                         increase               10   
                                         reduce                 10   
                                     D   auto_grad              10   
                                         increase               10   
                                         reduce                 10   

                                                       mean_lead_time  \
optimizer condition                  arm intervention                   
adam      drift_forecastable         C   auto_grad             5.3000   
                                         increase              5.3000   
                                         reduce                5.3000   
                                     D   auto_grad             2.2000   
                                         increase              2.2000   
                                         re

In [6]:
print("CHECK 1 -- does Arm D trigger earlier than Arm C (forecastable conditions)?")
for optimizer_name in OPTIMIZERS:
    for family in ("drift", "instability"):
        cond = f"{family}_forecastable"
        d = get(optimizer_name, cond, "C", "reduce", "mean_lead_time") - get(optimizer_name, cond, "D", "reduce", "mean_lead_time")
        print(f"  [{optimizer_name:4s}] {cond}: lead-time diff (C-D) = {d:.2f} steps")
print()

print("CHECK 2 -- winner by peak loss (drift) / steady-state loss (instability), reduce vs increase:")
for optimizer_name in OPTIMIZERS:
    peak_reduce = get(optimizer_name, "drift_forecastable", "C", "reduce", "mean_peak_loss")
    peak_increase = get(optimizer_name, "drift_forecastable", "C", "increase", "mean_peak_loss")
    ss_reduce = get(optimizer_name, "instability_unforecastable", "C", "reduce", "mean_steady_state_loss")
    ss_increase = get(optimizer_name, "instability_unforecastable", "C", "increase", "mean_steady_state_loss")
    drift_winner = "reduce" if peak_reduce < peak_increase else "increase"
    instab_winner = "reduce" if ss_reduce < ss_increase else "increase"
    print(f"  [{optimizer_name:4s}] drift_forecastable peak-loss winner = {drift_winner:8s} (reduce={peak_reduce:.3f}, increase={peak_increase:.3f})")
    print(f"  [{optimizer_name:4s}] instability_unforecastable steady-state winner = {instab_winner:8s} (reduce={ss_reduce:.4f}, increase={ss_increase:.4f})")
print()

print("CHECK 3 -- auto_grad diagnostic accuracy (chose 'increase' for drift, 'reduce' for instability):")
expected = {"drift_forecastable": "increase", "drift_unforecastable": "increase",
            "instability_forecastable": "reduce", "instability_unforecastable": "reduce"}
for optimizer_name in OPTIMIZERS:
    auto_rows = df[(df["optimizer"] == optimizer_name) & (df["intervention"] == "auto_grad") & df["trigger_step"].notna()]
    n_correct = sum((auto_rows["condition"] == c) & (auto_rows["active_intervention"] == expected[c]) for c in expected).sum() if len(auto_rows) else 0
    correct_mask = auto_rows.apply(lambda row: row["active_intervention"] == expected[row["condition"]], axis=1)
    acc = correct_mask.mean() if len(auto_rows) else float("nan")
    n_triggered = len(auto_rows)
    n_total = len(CONDITIONS) * 2 * len(SEEDS)
    print(f"  [{optimizer_name:4s}] auto_grad: {n_triggered}/{n_total} triggered, accuracy given triggered = {acc*100:.1f}%")


CHECK 1 -- does Arm D trigger earlier than Arm C (forecastable conditions)?
  [sgd ] drift_forecastable: lead-time diff (C-D) = 3.10 steps
  [sgd ] instability_forecastable: lead-time diff (C-D) = 7.90 steps
  [adam] drift_forecastable: lead-time diff (C-D) = 3.10 steps
  [adam] instability_forecastable: lead-time diff (C-D) = 28.80 steps

CHECK 2 -- winner by peak loss (drift) / steady-state loss (instability), reduce vs increase:
  [sgd ] drift_forecastable peak-loss winner = increase (reduce=3.157, increase=1.546)
  [sgd ] instability_unforecastable steady-state winner = reduce   (reduce=1.6954, increase=2.1979)
  [adam] drift_forecastable peak-loss winner = increase (reduce=4.883, increase=1.827)
  [adam] instability_unforecastable steady-state winner = reduce   (reduce=0.0135, increase=0.0274)

CHECK 3 -- auto_grad diagnostic accuracy (chose 'increase' for drift, 'reduce' for instability):
  [sgd ] auto_grad: 80/80 triggered, accuracy given triggered = 93.8%
  [adam] auto_grad: 79

## 5. Interpretation

In [7]:
sgd_peak_reduce = get("sgd", "drift_forecastable", "C", "reduce", "mean_peak_loss")
sgd_peak_increase = get("sgd", "drift_forecastable", "C", "increase", "mean_peak_loss")
adam_peak_reduce = get("adam", "drift_forecastable", "C", "reduce", "mean_peak_loss")
adam_peak_increase = get("adam", "drift_forecastable", "C", "increase", "mean_peak_loss")

sgd_ss_reduce = get("sgd", "instability_unforecastable", "C", "reduce", "mean_steady_state_loss")
sgd_ss_increase = get("sgd", "instability_unforecastable", "C", "increase", "mean_steady_state_loss")
adam_ss_reduce = get("adam", "instability_unforecastable", "C", "reduce", "mean_steady_state_loss")
adam_ss_increase = get("adam", "instability_unforecastable", "C", "increase", "mean_steady_state_loss")

sgd_drift_winner = "reduce" if sgd_peak_reduce < sgd_peak_increase else "increase"
adam_drift_winner = "reduce" if adam_peak_reduce < adam_peak_increase else "increase"
sgd_instab_winner = "reduce" if sgd_ss_reduce < sgd_ss_increase else "increase"
adam_instab_winner = "reduce" if adam_ss_reduce < adam_ss_increase else "increase"

drift_agrees = (sgd_drift_winner == adam_drift_winner)
instab_agrees = (sgd_instab_winner == adam_instab_winner)

sgd_d_lead = get("sgd", "drift_forecastable", "C", "reduce", "mean_lead_time") - get("sgd", "drift_forecastable", "D", "reduce", "mean_lead_time")
adam_d_lead = get("adam", "drift_forecastable", "C", "reduce", "mean_lead_time") - get("adam", "drift_forecastable", "D", "reduce", "mean_lead_time")

interpretation = f'''
R4 -- DOES THE CORE PATTERN HOLD UNDER ADAM, NOT JUST SGD?

Anticipatory-vs-reactive lead time (drift_forecastable, C-D difference, positive = D earlier):
  SGD:  {sgd_d_lead:.2f} steps
  Adam: {adam_d_lead:.2f} steps
{"Both optimizers show D triggering earlier -- the anticipatory-vs-reactive timing advantage is not SGD-specific." if (sgd_d_lead > 0 and adam_d_lead > 0) else "The lead-time advantage does NOT hold under both optimizers -- this needs investigating before treating the anticipatory claim as optimizer-general. (NOTE: this check has the same unresolved false-alarm-rate caveat as R8/R3 -- lead time alone, without matching false-alarm rate, is not yet a clean comparison under either optimizer.)"}

Drift wants "increase" -- winner by peak loss:
  SGD:  {sgd_drift_winner} (reduce={sgd_peak_reduce:.3f}, increase={sgd_peak_increase:.3f})
  Adam: {adam_drift_winner} (reduce={adam_peak_reduce:.3f}, increase={adam_peak_increase:.3f})

Instability wants "reduce" -- winner by steady-state loss:
  SGD:  {sgd_instab_winner} (reduce={sgd_ss_reduce:.4f}, increase={sgd_ss_increase:.4f})
  Adam: {adam_instab_winner} (reduce={adam_ss_reduce:.4f}, increase={adam_ss_increase:.4f})

AGREEMENT CHECK: drift winner {"AGREES" if drift_agrees else "DISAGREES"} between SGD and Adam.
                  instability winner {"AGREES" if instab_agrees else "DISAGREES"} between SGD and Adam.

{"R4 RESULT: the core diagnosis-must-match-disturbance pattern (increase for drift, reduce for instability) is CONFIRMED under Adam as well as SGD -- not an artifact of plain-SGD's simple, unpreconditioned response to LR scaling. This is meaningful evidence the pattern reflects something about the disturbance types themselves, not the specific optimizer." if (drift_agrees and instab_agrees) else "R4 RESULT: the core pattern does NOT fully replicate under Adam -- at least one of the drift/instability winners flips between optimizers. This means the 'increase for drift, reduce for instability' finding, as currently demonstrated, may be partly SGD-specific (plausibly because Adam's per-parameter adaptive step size already partially compensates for gradient-noise inflation via its second-moment normalization, changing how a fixed reduce/increase multiplier interacts with the underlying instability). This narrows the claim and should be reported as an open optimizer-dependence, not swept under the qualitative-check caveat."}

CAVEAT: base_lr and the reduce/increase multiplicative factors (0.3x/3.0x) were tuned
(informally, by earlier trial and error) around SGD's dynamics, then reused unchanged
for Adam with only the base_lr rescaled (0.1 -> 0.01) -- the 0.3x/3.0x factors
themselves were NOT re-tuned for Adam. A factor that works well for SGD is not
guaranteed to be well-scaled for Adam's adaptive step size; a null or reversed result
here could reflect an under/over-tuned intervention magnitude for Adam rather than a
genuine optimizer-dependence of the underlying phenomenon. This is a real limitation of
this specific test, not resolved here -- flagged for a possible follow-up (a small
sweep of Adam-specific reduce/increase factors) if this result is surprising.

Same Stage A caveats as all prior notebooks this session (n={len(SEEDS)} seeds, no
pre-registered effect sizes, not the frozen Stage D experiment).
'''
print(interpretation)



R4 -- DOES THE CORE PATTERN HOLD UNDER ADAM, NOT JUST SGD?

Anticipatory-vs-reactive lead time (drift_forecastable, C-D difference, positive = D earlier):
  SGD:  3.10 steps
  Adam: 3.10 steps
Both optimizers show D triggering earlier -- the anticipatory-vs-reactive timing advantage is not SGD-specific.

Drift wants "increase" -- winner by peak loss:
  SGD:  increase (reduce=3.157, increase=1.546)
  Adam: increase (reduce=4.883, increase=1.827)

Instability wants "reduce" -- winner by steady-state loss:
  SGD:  reduce (reduce=1.6954, increase=2.1979)
  Adam: reduce (reduce=0.0135, increase=0.0274)

AGREEMENT CHECK: drift winner AGREES between SGD and Adam.
                  instability winner AGREES between SGD and Adam.

R4 RESULT: the core diagnosis-must-match-disturbance pattern (increase for drift, reduce for instability) is CONFIRMED under Adam as well as SGD -- not an artifact of plain-SGD's simple, unpreconditioned response to LR scaling. This is meaningful evidence the pattern